In [2]:
import pandas as pd
import requests
from pathlib import Path
import sys

ROOT = Path().resolve().parents[1] 
sys.path.append(str(ROOT))

from data_gathering.dimLocation.initializer import getLocationKey



In [3]:
file_path = ROOT / "data" / "DEPI_HOGENT_mobiliteitsbevraging_2024.csv"

In [4]:
file_path = ROOT / "data" / "DEPI_HOGENT_mobiliteitsbevraging_2024.csv"

if not file_path.exists():
    print(f"FOUT: Bestand niet gevonden op: {file_path}")
else:
    df = pd.read_csv(file_path, sep=";", encoding='unicode_escape')

In [5]:
df.head()

,StartDate,EndDate,Status,Progress,Duration__in_seconds_,Finished,RecordedDate,ResponseId,RecipientLastName,RecipientFirstName,...,infrastructuur_3,leeftijd,functie,functie_6_TEXT,werkrooster,geslacht,gender,gender_4_TEXT,postcode,kansen
0,23/09/2024 2:04,24/09/2024 7:47,0,760,1070330,0,1/10/2024 7:47,R_1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23/09/2024 4:13,24/09/2024 7:48,0,760,993070,0,1/10/2024 7:48,R_2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20/09/2024 6:08,24/09/2024 8:16,0,60,3532910,0,1/10/2024 8:16,R_3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1/10/2024 12:09,1/10/2024 12:15,0,1000,4090,10,1/10/2024 12:16,R_4,NaN,NaN,...,30.0,450.0,10.0,NaN,10.0,20.0,20.0,NaN,90000.0,NaN
4,1/10/2024 12:14,1/10/2024 12:23,0,1000,5200,10,1/10/2024 12:23,R_5,NaN,NaN,...,10.0,390.0,10.0,NaN,10.0,20.0,20.0,NaN,93100.0,Fietsleasing


### Alleen relevante data bijhouden

In [6]:
df = df[['RecordedDate', 'ResponseId', 'LocationLatitude', 'LocationLongitude','werkplek', 'pendeltijd', 'pendelafstand', 'vervoermiddel', 'Finished', 'functie', 'werk__', 'thuiswerk']]

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 665 entries, 0 to 664
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RecordedDate       665 non-null    str    
 1   ResponseId         665 non-null    str    
 2   LocationLatitude   530 non-null    float64
 3   LocationLongitude  530 non-null    float64
 4   werkplek           650 non-null    float64
 5   pendeltijd         605 non-null    float64
 6   pendelafstand      605 non-null    float64
 7   vervoermiddel      605 non-null    float64
 8   Finished           665 non-null    int64  
 9   functie            524 non-null    float64
 10  werk__             605 non-null    float64
 11  thuiswerk          605 non-null    float64
dtypes: float64(9), int64(1), str(2)
memory usage: 62.5 KB


In [11]:
df.head(10)

,RecordedDate,ResponseId,LocationLatitude,LocationLongitude,werkplek,pendeltijd,pendelafstand,vervoermiddel,Finished,functie,werk__,thuiswerk,DateTime,TimeKey
0,1/10/2024 7:47,R_1,NaN,NaN,70.0,120.0,30.0,100.0,0,NaN,60.0,40.0,2024-01-10 07:47:00,747
1,1/10/2024 7:48,R_2,NaN,NaN,50.0,120.0,30.0,150.0,0,NaN,40.0,40.0,2024-01-10 07:48:00,748
2,1/10/2024 8:16,R_3,NaN,NaN,80.0,NaN,NaN,NaN,0,NaN,NaN,NaN,2024-01-10 08:16:00,816
3,1/10/2024 12:16,R_4,51047.0,37206.0,80.0,200.0,60.0,100.0,10,10.0,60.0,40.0,2024-01-10 12:16:00,1216
4,1/10/2024 12:23,R_5,51047.0,37206.0,70.0,550.0,400.0,10.0,10,10.0,60.0,40.0,2024-01-10 12:23:00,1223
5,1/10/2024 12:24,R_6,51096.0,38319.0,80.0,550.0,250.0,100.0,10,70.0,60.0,30.0,2024-01-10 12:24:00,1224
6,1/10/2024 12:35,R_7,508854.0,33581.0,80.0,100.0,20.0,100.0,10,10.0,60.0,20.0,2024-01-10 12:35:00,1235
7,1/10/2024 13:12,R_8,508847.0,45049.0,70.0,450.0,250.0,100.0,10,20.0,60.0,30.0,2024-01-10 13:12:00,1312
8,1/10/2024 13:33,R_9,509347.0,41296.0,80.0,600.0,360.0,100.0,10,40.0,60.0,10.0,2024-01-10 13:33:00,1333
9,1/10/2024 13:59,R_10,509475.0,31205.0,20.0,750.0,360.0,10.0,10,20.0,60.0,30.0,2024-01-10 13:59:00,1359


In [10]:
df['DateTime'] = pd.to_datetime(df['RecordedDate'], format='mixed')

# 2. Formatteer naar HHMM als string ('%H%M')
df['TimeKey'] = df['DateTime'].dt.strftime('%H%M')

# 3. Omzetten naar een getal (integer)
df['TimeKey'] = df['TimeKey'].astype(int)

In [73]:
def corrigeer_lat(val):
    lengte = len(str(val).replace('.', ''))
    
    if lengte == 6: # Bijv. 5104.0 (5 cijfers)
        return val / 1000
    elif lengte == 7: # Bijv. 50885.0 (6 cijfers)
        return val / 10000
    else:
        return val 

In [74]:
df['LocationLatitude'] = df['LocationLatitude'].apply(corrigeer_lat)
df['LocationLongitude'] = df['LocationLongitude'] / 10000

### Add geolocation

In [75]:
def get_vlaanderen_data(lat, lon):
    url = f"https://geo.api.vlaanderen.be/geolocation/v4/Location?latlon={lat},{lon}"
    try:
        r = requests.get(url, timeout=5)
        res = r.json().get('LocationResult', []) # print als je de res is wilt zien van de api
        if res:
            return str(res[0].get('Zipcode')) # enkel postcode eruit halen
    except:
        pass
    return None

In [76]:
ZIPCODE_FILE = ROOT / "data" / "zipcodes_num_nl_2025.xls"

In [77]:
df_zip = pd.read_excel(ZIPCODE_FILE)

In [78]:
df_zip.info()

<class 'pandas.DataFrame'>
RangeIndex: 2764 entries, 0 to 2763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Postcode       2764 non-null   int64
 1   Plaatsnaam     2764 non-null   str  
 2   Deelgemeente   2720 non-null   str  
 3   Hoofdgemeente  2764 non-null   str  
 4   Provincie      2720 non-null   str  
dtypes: int64(1), str(4)
memory usage: 108.1 KB


In [79]:
for index, row in df.iterrows():
    print(f"plaats {index} ophalen van de {len(df)}")
    lat, lon = row['LocationLatitude'], row['LocationLongitude']

    if pd.isna(lat) or pd.isna(lon):
        print(f"geen lat en/of longtitude gevonden voor plaats {index}")
        continue
    
    pc = get_vlaanderen_data(lat, lon)
    
    if pc:
        match_excel = df_zip[df_zip['Postcode'] == int(pc)]
        
        if not match_excel.empty:
            mainMunicipality = match_excel.iloc[0]['Hoofdgemeente'].capitalize()
            municipality = match_excel.iloc[0]['Plaatsnaam'].capitalize()
            province = match_excel.iloc[0]['Provincie'].title()
            
            df.at[index, "LocationKey"] = getLocationKey(postalCode=pc, MainMunicipality=mainMunicipality, Municipality=municipality, Province=province)
            print(f"{municipality} met postcode {pc} has been found!")
        else:
            print(f"Postcode {pc} gevonden via API, maar staat niet in de Excel lijst.")

plaats 0 ophalen van de 665
geen lat en/of longtitude gevonden voor plaats 0
plaats 1 ophalen van de 665
geen lat en/of longtitude gevonden voor plaats 1
plaats 2 ophalen van de 665
geen lat en/of longtitude gevonden voor plaats 2
plaats 3 ophalen van de 665
Gent met postcode 9000 has been found!
plaats 4 ophalen van de 665
Gent met postcode 9000 has been found!
plaats 5 ophalen van de 665
Beervelde met postcode 9080 has been found!
plaats 6 ophalen van de 665
Desselgem met postcode 8792 has been found!
plaats 7 ophalen van de 665
Nossegem met postcode 1930 has been found!
plaats 8 ophalen van de 665
Baardegem met postcode 9310 has been found!
plaats 9 ophalen van de 665
Beveren met postcode 8800 has been found!
plaats 10 ophalen van de 665
Gontrode met postcode 9090 has been found!
plaats 11 ophalen van de 665
Gent met postcode 9000 has been found!
plaats 12 ophalen van de 665
Sint-amandsberg met postcode 9040 has been found!
plaats 13 ophalen van de 665
Ertvelde met postcode 9940 has

In [80]:
df.columns

Index(['RecordedDate', 'ResponseId', 'LocationLatitude', 'LocationLongitude',
       'werkplek', 'pendeltijd', 'pendelafstand', 'vervoermiddel', 'Finished',
       'functie', 'werk__', 'thuiswerk', 'LocationKey'],
      dtype='str')

In [81]:
for i in df.columns[4:]:
    df[i] = df[i] / 10

In [82]:
import json

with open(ROOT / "data" / "DEPI_meta_dict_HOGENT_mobiliteitsbevraging_2024.json", 'r', encoding='utf-8') as file:
    metadata = json.load(file)

In [ ]:
def apply_mapping(df, column_name, map_dict):
    # Zorg dat de sleutels in de map_dict floats zijn, 
    # omdat de data in het dataframe vaak floats zijn.
    cleaned_map = {float(k): v for k, v in map_dict.items()}
    df[column_name] = df[column_name].map(cleaned_map)
    return df

In [84]:
for i in df.columns[4:]:
    if i in metadata['variable_value_labels']:
        df = apply_mapping(df, i, metadata['variable_value_labels'][i])

In [85]:
df.head()

,RecordedDate,ResponseId,LocationLatitude,LocationLongitude,werkplek,pendeltijd,pendelafstand,vervoermiddel,Finished,functie,werk__,thuiswerk,LocationKey
0,1/10/2024 7:47,R_1,NaN,NaN,Mercator,12.0,3.0,Fiets of elektrische fiets (speed pedelec inbe...,False,NaN,100% (voltijdse betrekking),Thuiswerk 2 dagen/week,NaN
1,1/10/2024 7:48,R_2,NaN,NaN,Ledeganck,12.0,3.0,Trein,False,NaN,60% - 79%,Thuiswerk 2 dagen/week,NaN
2,1/10/2024 8:16,R_3,NaN,NaN,"Schoonmeersen Noord (gebouwen B,C,D, E, P, S)",NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
3,1/10/2024 12:16,R_4,51.047,3.7206,"Schoonmeersen Noord (gebouwen B,C,D, E, P, S)",20.0,6.0,Fiets of elektrische fiets (speed pedelec inbe...,True,Administratief personeel,100% (voltijdse betrekking),Thuiswerk 2 dagen/week,246.3
4,1/10/2024 12:23,R_5,51.047,3.7206,Mercator,55.0,40.0,"Wagen, bestelwagen of vrachtwagen alleen of me...",True,Administratief personeel,100% (voltijdse betrekking),Thuiswerk 2 dagen/week,246.3


In [ ]:
df = df.rename(columns={
    'RecordedDate': 'RecordDate',
    'ResponseId': 'ResponseID',
    'LocationLatitude': 'Latitude',
    'LocationLongitude': 'Longtitude',
    'werkplek': 'WorkPlace',
    'pendeltijd': 'TravelTime',
    'pendelafstand': 'TravelDistance',
    'vervoermiddel': 'TravelType',
    "functie": "WorkoFunction",
    "werk__": "WorkRegime",
    'thuiswerk': "HomeWork"
})

In [87]:
df['RecordDate'] = pd.to_datetime(df['RecordDate'], format="%d/%m/%Y %H:%M").dt.strftime('%Y%m%d')

In [88]:
df["RecordDate"] = df["RecordDate"].astype("Int64")

In [92]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 665 entries, 0 to 664
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   RecordDate      665 non-null    Int64  
 1   ResponseID      665 non-null    str    
 2   Latitude        530 non-null    float64
 3   Longtitude      530 non-null    float64
 4   WorkPlace       650 non-null    str    
 5   TravelTime      605 non-null    float64
 6   TravelDistance  605 non-null    float64
 7   TravelType      605 non-null    str    
 8   Finished        665 non-null    str    
 9   Function        524 non-null    str    
 10  WorkRegime      605 non-null    str    
 11  HomeWork        605 non-null    str    
 12  LocationKey     497 non-null    float64
dtypes: Int64(1), float64(5), str(7)
memory usage: 68.3 KB


In [90]:
df.head()

,RecordDate,ResponseID,Latitude,Longtitude,WorkPlace,TravelTime,TravelDistance,TravelType,Finished,functie,WorkRegime,HomeWork,LocationKey
0,20241001,R_1,NaN,NaN,Mercator,12.0,3.0,Fiets of elektrische fiets (speed pedelec inbe...,False,NaN,100% (voltijdse betrekking),Thuiswerk 2 dagen/week,NaN
1,20241001,R_2,NaN,NaN,Ledeganck,12.0,3.0,Trein,False,NaN,60% - 79%,Thuiswerk 2 dagen/week,NaN
2,20241001,R_3,NaN,NaN,"Schoonmeersen Noord (gebouwen B,C,D, E, P, S)",NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
3,20241001,R_4,51.047,3.7206,"Schoonmeersen Noord (gebouwen B,C,D, E, P, S)",20.0,6.0,Fiets of elektrische fiets (speed pedelec inbe...,True,Administratief personeel,100% (voltijdse betrekking),Thuiswerk 2 dagen/week,246.3
4,20241001,R_5,51.047,3.7206,Mercator,55.0,40.0,"Wagen, bestelwagen of vrachtwagen alleen of me...",True,Administratief personeel,100% (voltijdse betrekking),Thuiswerk 2 dagen/week,246.3
